## Recommender Systems with PySpark: Movie Lens Dataset

### Collaborative Filtering: Alternating Least Squares (ALS)

- numBlocks (-1 imply auto-config)
- rank
- iterations
- lambda: regularization
- implicitPref
- alpha

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [2]:
spark = SparkSession.builder.appName("movielens").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 15:31:23 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/07/06 15:31:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 15:31:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv('ratings.csv', inferSchema=True, header=True, sep=';')

In [4]:
df.show()

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
|     1|   1197|     3|978302268|
|     1|   1287|     5|978302039|
|     1|   2804|     5|978300719|
|     1|    594|     4|978302268|
|     1|    919|     4|978301368|
|     1|    595|     5|978824268|
|     1|    938|     4|978301752|
|     1|   2398|     4|978302281|
|     1|   2918|     4|978302124|
|     1|   1035|     5|978301753|
|     1|   2791|     4|978302188|
|     1|   2687|     3|978824268|
|     1|   2018|     4|978301777|
|     1|   3105|     5|978301713|
|     1|   2797|     4|978302039|
+------+-------+------+---------+
only showing top 20 rows


In [5]:
df = df.drop('timestamp')

In [6]:
df.show()

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   1193|     5|
|     1|    661|     3|
|     1|    914|     3|
|     1|   3408|     4|
|     1|   2355|     5|
|     1|   1197|     3|
|     1|   1287|     5|
|     1|   2804|     5|
|     1|    594|     4|
|     1|    919|     4|
|     1|    595|     5|
|     1|    938|     4|
|     1|   2398|     4|
|     1|   2918|     4|
|     1|   1035|     5|
|     1|   2791|     4|
|     1|   2687|     3|
|     1|   2018|     4|
|     1|   3105|     5|
|     1|   2797|     4|
+------+-------+------+
only showing top 20 rows


In [7]:
# Checking for null values

from pyspark.sql.functions import col, sum

# Sum up boolean true values (cast as 1) for each column
null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.show()

[Stage 4:=============================>                             (1 + 1) / 2]

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     0|      0|     0|
+------+-------+------+



In [8]:
# Checking for NaN values

from pyspark.sql.functions import col, sum, isnan

# Sum up boolean true values (cast as 1) for each column
nan_counts = df.select([sum(isnan(col(c)).cast("int")).alias(c) for c in df.columns])
nan_counts.show()

[Stage 7:=============================>                             (1 + 1) / 2]

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     0|      0|     0|
+------+-------+------+



In [9]:
df.describe().show()

[Stage 10:=============================>                            (1 + 1) / 2]

+-------+-----------------+------------------+------------------+
|summary|           userId|           movieId|            rating|
+-------+-----------------+------------------+------------------+
|  count|          1000209|           1000209|           1000209|
|   mean|3024.512347919285|1865.5398981612843| 3.581564453029317|
| stddev|1728.412694899932|1096.0406894572586|1.1171018453732577|
|    min|                1|                 1|                 1|
|    max|             6040|              3952|                 5|
+-------+-----------------+------------------+------------------+



In [10]:
(train, test) = df.randomSplit([0.7, 0.3], seed=42)

In [20]:
als = ALS(maxIter=5, regParam=0.01, userCol='userId', itemCol='movieId', ratingCol='rating', coldStartStrategy="drop")

In [21]:
model = als.fit(train)

In [22]:
pred = model.transform(test)

In [23]:
pred.show()

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|     11|     5| 4.4701605|
|   148|     17|     4|  3.585598|
|   148|     89|     4|  3.510471|
|   148|    107|     4| 4.1727157|
|   148|    165|     3| 3.9695714|
|   148|    185|     3| 3.8559942|
|   148|    204|     3| 3.3690631|
|   148|    227|     4| 3.3981037|
|   148|    258|     3| 3.6384904|
|   148|    261|     3| 3.4959738|
|   148|    293|     3| 3.7213876|
|   148|    300|     2| 3.4058251|
|   148|    329|     3| 4.0198913|
|   148|    332|     3| 2.7682967|
|   148|    349|     5|  4.132382|
|   148|    350|     4| 3.5926924|
|   148|    351|     3| 3.4137974|
|   148|    361|     4|  3.961548|
|   148|    364|     5| 4.5817814|
|   148|    368|     5|  4.391815|
+------+-------+------+----------+
only showing top 20 rows


In [24]:
evals = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')

In [25]:
rmse = evals.evaluate(pred)
print(f"RMSE: {rmse}")

[Stage 245:>                                                        (0 + 2) / 2]

RMSE: 0.9107031734099926


RMSE describe our error in terms of the stars rating column. 

In [26]:
user_1 = test.filter(test['userId'] == 1).select(['movieId', 'userId'])

In [27]:
user_1.show()

[Stage 247:>                                                        (0 + 1) / 1]

+-------+------+
|movieId|userId|
+-------+------+
|    150|     1|
|    588|     1|
|    595|     1|
|    608|     1|
|    783|     1|
|    914|     1|
|    919|     1|
|   1029|     1|
|   1097|     1|
|   1197|     1|
|   1207|     1|
|   1545|     1|
|   1566|     1|
|   1721|     1|
|   1907|     1|
|   1962|     1|
|   2018|     1|
|   2340|     1|
|   2687|     1|
|   2692|     1|
+-------+------+
only showing top 20 rows


In [28]:
rec = model.transform(user_1)

In [29]:
rec.orderBy('prediction', ascending=False).show()

+-------+------+----------+
|movieId|userId|prediction|
+-------+------+----------+
|   2340|     1| 5.2555704|
|    783|     1|   5.17705|
|    919|     1|  5.042333|
|   2804|     1| 5.0286617|
|   1207|     1| 4.8632097|
|   1029|     1|  4.784983|
|   2687|     1| 4.7036805|
|   2791|     1| 4.6852593|
|   1097|     1| 4.6843047|
|   2018|     1|  4.645855|
|   1721|     1| 4.5266895|
|    595|     1| 4.5057755|
|    914|     1| 4.4793534|
|   2797|     1| 4.4016275|
|   1907|     1| 4.3296747|
|    588|     1| 4.3232684|
|    150|     1| 4.2741327|
|   3105|     1|  4.220649|
|   1197|     1|  4.205823|
|   1962|     1| 4.0874267|
+-------+------+----------+
only showing top 20 rows
